# Baselines: aleatoria y FIFO

**He(RL)l Kitchen v2.** Referencias de desempeño contra las que se comparan DQN y PPO (`docs/env_design.md`, sección 4.3). No se entrenan: evaluarlas cuesta segundos.

- **Aleatoria:** acción uniforme. Cota inferior.
- **FIFO ingenua:** trabaja solo sobre la orden más antigua: la prepara, la pone al fuego, prueba cuando se ve casi lista y sirve o descarta según la prueba.

Se evalúan con las **seeds de prueba** (`10000 + i`) en todos los escenarios de la sección 4.6.

**Cómo correrlo**

- **Local:** con el entorno virtual activo, abrir este notebook desde `notebooks/` y ejecutar todo.
- **Google Colab:** abrir el notebook (desde GitHub o subiéndolo) y ejecutar todo; la primera celda clona el repo e instala las dependencias.

Las tablas y figuras se guardan en `results/` (o en la carpeta de la variable de entorno `KITCHEN_NB_OUTPUT_DIR`, que usan los tests).

In [ ]:
# Arranque: en Colab clona el repo; en local no hace nada extra.
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("/content/PROY-RL").exists():
    subprocess.run(["git", "clone", "https://github.com/paulabaal12/PROY-RL.git",
                    "/content/PROY-RL"], check=True)
sys.path.insert(0, "/content/PROY-RL/notebooks" if "google.colab" in sys.modules else ".")

from _setup import setup
ROOT = setup()
print("Raíz del repo:", ROOT)

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import pandas as pd

# Episodios por configuración. Los tests lo bajan con KITCHEN_NB_EPISODES.
EPISODES = int(os.environ.get("KITCHEN_NB_EPISODES", 100))
OUT = Path(os.environ.get("KITCHEN_NB_OUTPUT_DIR", "results"))
FIGURES, TABLES = OUT / "figures", OUT / "tables"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("Episodios por configuración:", EPISODES)

from agents.baselines import FifoHeuristicPolicy, RandomPolicy
from envs.kitchen_env import KitchenEnv
from experiments import calibrate
from experiments.evaluate import evaluate_scenarios, summarize, to_markdown

WINDOW = 4

## Un episodio en ASCII

Primeros 30 pasos de la FIFO en la cocina `medium`. Cada paso muestra la acción, el nivel **real** de cocción de cada estación (con una barra), lo que **ve** el agente (lectura con ruido), la última prueba, la ventana visible y los eventos.

In [ ]:
env = KitchenEnv(window=WINDOW, kitchen="medium", render_mode="ansi")
policy = FifoHeuristicPolicy(WINDOW)
obs, _ = env.reset(seed=0)
total = 0.0
for _ in range(30):
    obs, reward, _, _, _ = env.step(policy(obs))
    total += reward
    print(env.render())
    print(f"Recompensa: {reward:+.2f} | acumulada: {total:+.2f}\n")

## Evaluación por escenario

In [ ]:
results = {
    "random": evaluate_scenarios(lambda env: RandomPolicy(env.action_space.n, seed=0),
                                 n_episodes=EPISODES, env_kwargs={"window": WINDOW}),
    "fifo": evaluate_scenarios(lambda env: FifoHeuristicPolicy(WINDOW),
                               n_episodes=EPISODES, env_kwargs={"window": WINDOW}),
}
episodes = pd.concat([df.assign(policy=name) for name, df in results.items()], ignore_index=True)
summary = summarize(results)
episodes.to_csv(TABLES / "baselines_episodes.csv", index=False)
summary.to_csv(TABLES / "baselines_summary.csv", index=False)
(TABLES / "baselines_summary.md").write_text(to_markdown(summary) + "\n", encoding="utf-8")
summary

In [ ]:
fig = calibrate.plot_difficulty(episodes, out_dir=None)
fig.savefig(FIGURES / "baselines_by_scenario.png", dpi=150, facecolor=fig.get_facecolor())

## Conclusiones

*Pendiente: se completa después de ejecutar el notebook con todos los episodios, una vez calibrado el ambiente (Task 9 del plan).*